# 信噪 · 4 路对比评估（Kaggle 瘦壳）

这个 notebook 只是 `evals/run_compare.py` 的 Kaggle 运行器。逻辑全部在那个 .py 里。

对比策略：
1. **deepseek-base** —— DeepSeek-V3 纯 LLM（无 RAG、无工具）
2. **deepseek-agent** —— 当前完整 agent（RAG + 5 工具 + injection guard）
3. **qwen-base** —— Qwen2.5-1.5B-Instruct 原生（无微调）
4. **qwen-lora** —— Qwen2.5-1.5B + 你刚训出来的 LoRA adapter

## 准备
1. 把仓库代码 + 训练好的 adapter（含 `adapter_config.json`）都作为 Kaggle Dataset 加进 input
2. 配置 Secrets：`DEEPSEEK_API_KEY`、`TAVILY_API_KEY`（左侧 Add-ons → Secrets）
3. Settings → GPU T4/P100；Internet = On

In [ ]:
# 1. 安装依赖
#
# 同训练 notebook：Kaggle 预装的 transformers/trl/datasets 太新，
# 直接 -U 升级会跟 LoRA adapter（用兼容 unsloth 的旧版 transformers 训出来的）打架。
# 这里走跟训练完全一致的安装路径，保证版本对齐：卸掉冲突包 → 装 unsloth。
#
# 装完点 Run → Restart session，再跑后续 cell。
!pip uninstall -y -q transformers trl datasets unsloth unsloth-zoo
!pip install -q unsloth bitsandbytes
# Agent 运行时依赖
!pip install -q openai python-dotenv tavily-python sentence-transformers rank-bm25

In [ ]:
# 2. 注入 Secrets + 找代码 / adapter 路径
import os, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["DEEPSEEK_API_KEY"] = secrets.get_secret("DEEPSEEK_API_KEY")
os.environ["TAVILY_API_KEY"] = secrets.get_secret("TAVILY_API_KEY")

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/repo")

# 找代码目录
code_hits = list(INPUT_ROOT.rglob("evals/run_compare.py"))
if code_hits:
    code_root = code_hits[0].parents[1]
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    shutil.copytree(code_root, WORK_DIR)
else:
    !git clone https://github.com/alchosyn/npc-dialogue-ai-agent.git $WORK_DIR
os.chdir(WORK_DIR)
print(f"工作目录: {WORK_DIR}")

# 找 adapter —— 优先选不在 checkpoint-* 子目录里的（那是中间存档）
all_hits = list(INPUT_ROOT.rglob("adapter_config.json"))
final_hits = [p for p in all_hits if "checkpoint-" not in str(p)]
adapter_hits = final_hits if final_hits else all_hits
assert adapter_hits, "找不到 LoRA adapter（adapter_config.json）"
ADAPTER_PATH = adapter_hits[0].parent
print(f"adapter: {ADAPTER_PATH}")
print(f"  (共扫到 {len(all_hits)} 个 adapter_config.json, 最终用上面这个)")

In [ ]:
# 3. 跑 4 路对比
import subprocess, sys
cmd = [
    sys.executable, "evals/run_compare.py",
    "--strategies", "all",
    "--qwen-lora-path", str(ADAPTER_PATH),
]
print("+ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 4. 显示报告 + 拷到 Kaggle Output 方便下载
import shutil
from IPython.display import Markdown, display
report = Path("evals/compare_report.md").read_text(encoding="utf-8")
display(Markdown(report))
shutil.copy("evals/compare_report.md", "/kaggle/working/compare_report.md")
shutil.copy("evals/compare_results.json", "/kaggle/working/compare_results.json")